# 02 - SPLADE: Sparse Retrieval

This notebook walks you through:

1. Installing the extra dependencies needed for SPLADE
2. Indexing all passages (encoding + storing sparse vectors in PostgreSQL)
3. Running search queries with SPLADE
4. Evaluating retrieval quality against the `qrels` ground truth

> **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
> `passages`, `queries`, and `qrels` tables are populated.

In [1]:
import sys
from pathlib import Path

# Resolve project root from notebook location
project_root = Path.cwd().resolve().parent
print(f"Project root: {project_root}")

# Ensure project root is importable so `src` package can be resolved from notebooks/
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Check for .env file at project root
env_path = project_root / ".env"
if env_path.exists():
    print(f"Found .env file: {env_path}")
else:
    print("No .env found at project root. Default values from config.py will be used.")
    print("You can create one manually if needed.")

Project root: /home/tim/Documents/Projet_Big_Data_IR
Found .env file: /home/tim/Documents/Projet_Big_Data_IR/.env


## 1) Install Python dependencies

Run this once per environment. 

In [2]:
import sys
import subprocess

requirements_file = project_root / "requirements.txt"
print(f"Installing dependencies from: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)

Installing dependencies from: /home/tim/Documents/Projet_Big_Data_IR/requirements.txt


Reason for being yanked: Backward compatibility bug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 4.8 MB/s  0:00:03 eta 0:00:010:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2


CompletedProcess(args=['/home/tim/Documents/Projet_Big_Data_IR/.venv/bin/python', '-m', 'pip', 'install', '-r', '/home/tim/Documents/Projet_Big_Data_IR/requirements.txt'], returncode=0)

## 2) Index passages with SPLADE

This step encodes every passage in the `passages` table and stores the resulting sparse vector (`{token: weight}` JSON) in the `splade` table.

**Estimated time:** ~1-3 min per 10 000 passages on CPU, much faster on GPU.

You can re-run safely — already-indexed passages are skipped.

### Optional fast path — import precomputed SPLADE vectors from CSV

If you already have a full `splade_data.csv`, you can load it directly into PostgreSQL and skip local encoding.

This is usually much faster than re-indexing with the model.

Expected CSV format:
- `passage_id` (BIGINT)
- `term_weights` (JSON text, e.g. `{"token": 1.23, ...}`)

> By default this cell truncates and fully replaces local SPLADE vectors.

In [ ]:
import time
from src.database.connection import get_connection

# Update this path depending on where your CSV is located. It should be the output of the SPLADE export script.
csv_path = project_root / "src" / "splade" / "splade_data.csv"
print(f"CSV path: {csv_path}")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

def format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds / 60:.2f} min"

def log_step(step_name: str, start_time: float, global_start: float) -> None:
    step_elapsed = time.time() - start_time
    total_elapsed = time.time() - global_start
    print(
        f"[{step_name}] done in {format_duration(step_elapsed)} "
        f"(total elapsed: {format_duration(total_elapsed)})",
        flush=True,
    )

conn = get_connection()
cur = conn.cursor()

try:
    t0 = time.time()
    print("Starting SPLADE CSV import...", flush=True)

    # Faster bulk-load settings for this transaction only
    step_t0 = time.time()
    cur.execute("SET LOCAL synchronous_commit = OFF")
    cur.execute("SET LOCAL work_mem = '256MB'")
    cur.execute("SET LOCAL maintenance_work_mem = '1GB'")
    log_step("Session setup", step_t0, t0)

    # Staging table for very fast COPY from CSV
    step_t0 = time.time()
    cur.execute("DROP TABLE IF EXISTS splade_import")
    cur.execute(
        """
        CREATE UNLOGGED TABLE splade_import (
            passage_id BIGINT,
            term_weights TEXT
        )
        """
    )
    log_step("Create staging table", step_t0, t0)

    # Bulk load CSV (with header)
    step_t0 = time.time()
    print("Starting COPY from CSV into staging table...", flush=True)
    with open(csv_path, "r", encoding="utf-8") as f:
        cur.copy_expert(
            """
            COPY splade_import (passage_id, term_weights)
            FROM STDIN
            WITH (FORMAT csv, HEADER true)
            """,
            f,
        )
    log_step("COPY into staging table", step_t0, t0)

    # Count imported rows and expected rows after FK-safe join
    step_t0 = time.time()
    cur.execute("SELECT COUNT(*) FROM splade_import")
    imported_row = cur.fetchone()
    if imported_row is None:
        raise RuntimeError("Could not read row count from splade_import")
    imported_rows = imported_row[0]
    cur.execute(
        """
        SELECT COUNT(*)
        FROM splade_import si
        JOIN passages p ON p.id = si.passage_id
        """
    )
    expected_row = cur.fetchone()
    if expected_row is None:
        raise RuntimeError("Could not read expected row count for splade import")
    expected_rows = expected_row[0]
    skipped_rows = imported_rows - expected_rows
    print(f"Rows copied to staging: {imported_rows}", flush=True)
    print(f"Rows matching passages: {expected_rows}", flush=True)
    if skipped_rows:
        print(f"Rows skipped because passage_id is missing: {skipped_rows}", flush=True)
    log_step("Validate staging rows", step_t0, t0)

    # Critical optimization: avoid maintaining the GIN index during mass insert
    step_t0 = time.time()
    cur.execute("DROP INDEX IF EXISTS idx_splade_term_weights")
    log_step("Drop GIN index", step_t0, t0)

    # Replace SPLADE content from imported data. Keep only rows whose passage_id exists in passages (FK safety)
    step_t0 = time.time()
    cur.execute("TRUNCATE TABLE splade")
    log_step("Truncate splade", step_t0, t0)

    step_t0 = time.time()
    print("Starting INSERT into splade...", flush=True)
    cur.execute(
        """
        INSERT INTO splade (passage_id, term_weights)
        SELECT si.passage_id, si.term_weights::jsonb
        FROM splade_import si
        JOIN passages p ON p.id = si.passage_id
        """
    )
    log_step("Insert into splade", step_t0, t0)

    # Rebuild index once, after load
    step_t0 = time.time()
    print("Rebuilding GIN index on splade.term_weights...", flush=True)
    cur.execute("CREATE INDEX idx_splade_term_weights ON splade USING gin (term_weights)")
    log_step("Rebuild GIN index", step_t0, t0)

    # Remove staging table
    step_t0 = time.time()
    cur.execute("DROP TABLE splade_import")
    log_step("Drop staging table", step_t0, t0)

    # Final validation before commit
    step_t0 = time.time()
    cur.execute("SELECT COUNT(*) FROM splade")
    inserted_row = cur.fetchone()
    if inserted_row is None:
        raise RuntimeError("Could not read final row count from splade")
    inserted = inserted_row[0]
    if inserted != expected_rows:
        raise RuntimeError(
            f"Row count mismatch: expected {expected_rows} rows in splade, found {inserted}"
        )
    log_step("Validate final row count", step_t0, t0)

    step_t0 = time.time()
    conn.commit()
    log_step("Commit transaction", step_t0, t0)

    elapsed = time.time() - t0
    print(f"Imported rows into splade: {inserted}", flush=True)
    print(f"Done in {format_duration(elapsed)}", flush=True)

except Exception:
    conn.rollback()
    raise
finally:
    cur.close()
    conn.close()

In [ ]:
from src.splade.indexer import index_passages

# Adjust batch sizes depending on your machine's RAM / VRAM
index_passages(batch_size=64, encoding_batch=32)

/home/tim/Documents/Projet_Big_Data_IR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-17 14:47:56,972 - INFO - Loading SPLADE model 'naver/splade-cocondenser-ensembledistil' on cpu ...
2026-03-17 14:47:57,612 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 14:47:57,671 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/config.json "HTTP/1.1 200 OK"
2026-03-17 14:47:57,847 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 14:47:57,890 - INFO - HTTP Request

2026-03-17 14:47:59,804 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocondenser-ensembledistil/discussions?p=0 "HTTP/1.1 200 OK"
2026-03-17 14:47:59,966 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocondenser-ensembledistil/commits/refs%2Fpr%2F6 "HTTP/1.1 200 OK"
2026-03-17 14:48:00,108 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/refs%2Fpr%2F6/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-03-17 14:48:00,267 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/refs%2Fpr%2F6/model.safetensors "HTTP/1.1 302 Found"


### Verify indexing

In [ ]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM splade", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample SPLADE entry (first passage):")
    sample = pd.read_sql_query(
        "SELECT passage_id, term_weights FROM splade ORDER BY passage_id LIMIT 1",
        conn
    )
    display(sample)
finally:
    conn.close()

## 3) Search with SPLADE

Two retrieval strategies:

- **GIN-based** (`search_gin`): uses the PostgreSQL GIN index to find candidate passages that share at least one term with the query, then scores them with a dot-product.
- **Brute-force** (`search_bruteforce`): scores every indexed passage. Slower but useful as a sanity check.

In [ ]:
from src.splade.search import search_gin

query = "what is the speed of light"

print(f"Query: '{query}'\n")
results = search_gin(query, top_k=10)

for i, r in enumerate(results, 1):
    print(f"[{i}] Score={r['score']:.4f}  Passage#{r['passage_id']}")
    print(f"    {r['text'][:150]}...\n")

In [ ]:
# Try your own queries
for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    results = search_gin(q, top_k=3)
    print(f"\nQuery: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r['score']:.4f} — {r['text'][:120]}...")

## 4) Evaluation — MRR@10

Compute **Mean Reciprocal Rank @ 10** using the ground-truth `qrels` table.

In [ ]:
import random
from src.database.connection import get_connection
from src.splade.search import search_gin
from src.splade.encoder import SpladeEncoder

conn = get_connection()
cursor = conn.cursor()

# Get queries that have at least one relevant passage (relevance = 1)
cursor.execute("""
    SELECT DISTINCT q.id, q.text
    FROM queries q
    JOIN qrels qr ON qr.query_id = q.id
    WHERE qr.relevance = 1
""")
eval_queries = cursor.fetchall()

# Sample a subset for faster evaluation
SAMPLE_SIZE = 20
if len(eval_queries) > SAMPLE_SIZE:
    random.seed(42)
    eval_queries = random.sample(eval_queries, SAMPLE_SIZE)

print(f"Evaluating on {len(eval_queries)} queries...\n")

encoder = SpladeEncoder()
reciprocal_ranks = []

for idx, (query_id, query_text) in enumerate(eval_queries):
    # Relevant passage IDs for this query
    cursor.execute(
        "SELECT passage_id FROM qrels WHERE query_id = %s AND relevance = 1",
        (query_id,)
    )
    relevant_ids = {row[0] for row in cursor.fetchall()}

    # Run SPLADE search
    results = search_gin(
        query_text, top_k=10, conn=conn, encoder=encoder, log_search=False
    )

    # Compute reciprocal rank
    rr = 0.0
    for rank, r in enumerate(results, 1):
        if r["passage_id"] in relevant_ids:
            rr = 1.0 / rank
            break
    reciprocal_ranks.append(rr)

    if (idx + 1) % 50 == 0:
        current_mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
        print(f"  [{idx+1}/{len(eval_queries)}] Running MRR@10 = {current_mrr:.4f}")

mrr_at_10 = sum(reciprocal_ranks) / len(reciprocal_ranks)
print(f"\n{'='*50}")
print(f"Final MRR@10 = {mrr_at_10:.4f}  ({len(eval_queries)} queries)")
print(f"{'='*50}")

cursor.close()
conn.close()

## Notes

- The first run downloads the model from Hugging Face (~400 MB). Set `HF_TOKEN` in `.env` if access is gated.
- For large-scale indexing, a GPU significantly speeds up encoding.
- The GIN search is fast for small-to-medium collections. For millions of passages, consider an inverted index in Elasticsearch or a dedicated sparse vector store.
- You can re-run indexing at any time; already-indexed passages are skipped automatically.